#Carga de información ya corregida


In [9]:
# ==============================================================================
# BLOQUE 7: ETL Y ACTUALIZACIÓN DEL DATASET MAESTRO (BASE ACUMULADA)
# ==============================================================================
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

display(Markdown("## 🏗️ Reconstrucción del Dataset Maestro (Formato Acumulado)"))

# 1. RUTAS DE ARCHIVOS
ruta_parquet_crudo = '/content/Consolidado_Suministros_Maestro.parquet'
ruta_excel = '/content/Bases_homologadas.xlsx'
ruta_2025 = '/content/2025_Primas_Fasecolda.xlsx'
ruta_2026 = '/content/2026_Primas_Fasecolda.xlsx'

# 2. CARGA Y UNIÓN
df_base = pd.read_parquet(ruta_parquet_crudo)
df_2025 = pd.read_excel(ruta_2025)
df_2026 = pd.read_excel(ruta_2026)
df = pd.concat([df_base, df_2025, df_2026], ignore_index=True)

df['FECHA'] = pd.to_datetime(df['FECHA'], format='%d/%m/%Y', errors='coerce')
df['AÑO'] = df['FECHA'].dt.year

df_homolog_ramos = pd.read_excel(ruta_excel, sheet_name='RAMO_FASECOLDA')
df_homolog_companias = pd.read_excel(ruta_excel, sheet_name='COMPANIA_FASECOLDA')
df_sucursal_regional = pd.read_excel(ruta_excel, sheet_name='SUCURSAL_REGIONAL')

# 3. HOMOLOGACIÓN
dict_ramos = dict(zip(df_homolog_ramos['RAMO'], df_homolog_ramos['HOMOLOG_RAMO']))
dict_companias = dict(zip(df_homolog_companias['COMPANIA'], df_homolog_companias['HOMOLOG_COMPANIA']))
dict_companias['ARL SURA'] = 'SURAMERICANA'
dict_companias['AXA COLPATRIA'] = 'AXA'
dict_companias['AXA COLPATRIA GENERALES'] = 'AXA'
dict_ramos['COLECTIVO Y GRUPO '] = 'GRUPO'

df['RAMOS'] = df['RAMOS'].apply(lambda x: dict_ramos.get(x, x))
df['COMPANIA'] = df['COMPANIA'].apply(lambda x: dict_companias.get(x, x))

df['CIUDAD'] = df['CIUDAD'].astype(str).str.strip().str.upper()
df_sucursal_regional['CIUDAD'] = df_sucursal_regional['CIUDAD'].astype(str).str.strip().str.upper()

df = df.merge(df_sucursal_regional[['CIUDAD', 'SUCURSAL', 'REGIONAL']], on='CIUDAD', how='left')
df['SUCURSAL'] = df['SUCURSAL'].fillna('SIN ASIGNAR')
df['REGIONAL'] = df['REGIONAL'].fillna('SIN ASIGNAR')

# 4. AGRUPACIÓN (MANTENIENDO EL ACUMULADO)
columnas_agrupacion = ['AÑO', 'FECHA', 'COMPANIA', 'RAMOS', 'CIUDAD', 'SUCURSAL', 'REGIONAL']
df_maestro = df.groupby(columnas_agrupacion)['TOTAL'].sum().reset_index()
df_maestro = df_maestro.sort_values(by=['AÑO', 'COMPANIA', 'RAMOS', 'CIUDAD', 'FECHA'])

# 5. GUARDADO
ruta_salida = '/content/Base_Maestra_Acumulada.parquet'
df_maestro.to_parquet(ruta_salida, index=False)

display(Markdown(f"### 💾 ¡Parquet Acumulado Guardado Exitosamente! (`{ruta_salida}`)"))

## 🏗️ Reconstrucción del Dataset Maestro (Formato Acumulado)

### 💾 ¡Parquet Acumulado Guardado Exitosamente! (`/content/Base_Maestra_Acumulada.parquet`)

#Cargue de la información

In [19]:
# ==============================================================================
# BLOQUE 0: CARGA DIRECTA DEL DATASET MAESTRO ACUMULADO
# ==============================================================================
# 1. Importar las librerías necesarias
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------
# PASO 1: DEFINIR LA RUTA DEL ARCHIVO
# ---------------------------------------------------------

# OPCIÓN A: Si subiste el archivo directamente a la sesión temporal de Colab (ícono de carpeta a la izquierda):
ruta_archivo = '/content/Base_Maestra_Acumulada.parquet'

# OPCIÓN B: Si el archivo está en tu Google Drive (Descomenta las siguientes 3 líneas si es tu caso):
# from google.colab import drive
# drive.mount('/content/drive')
# ruta_archivo = '/content/drive/MyDrive/TU_CARPETA_AQUI/Base_Maestra_Acumulada.parquet'

# ---------------------------------------------------------
# PASO 2: CARGAR LA BASE DE DATOS
# ---------------------------------------------------------

# Cargar el archivo parquet (Colab ya tiene el motor 'pyarrow' instalado por defecto)
df_maestro = pd.read_parquet(ruta_archivo)

# ---------------------------------------------------------
# PASO 3: INSPECCIÓN INICIAL (EDA Rápido)
# ---------------------------------------------------------

print("1. Información general de la base (Tipos de datos y nulos):")
df_maestro.info()

print("\n2. Resumen estadístico inicial de las variables numéricas:")
# Con display() la tabla se renderiza con el formato bonito e interactivo de Colab
display(df_maestro.describe().T)

print("\n3. Vista previa de las primeras 5 filas:")
display(df_maestro.head())

1. Información general de la base (Tipos de datos y nulos):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309347 entries, 0 to 309346
Data columns (total 8 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   AÑO       309347 non-null  int32         
 1   FECHA     309347 non-null  datetime64[ns]
 2   COMPANIA  309347 non-null  object        
 3   RAMOS     309347 non-null  object        
 4   CIUDAD    309347 non-null  object        
 5   SUCURSAL  309347 non-null  object        
 6   REGIONAL  309347 non-null  object        
 7   TOTAL     309347 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int32(1), object(5)
memory usage: 17.7+ MB

2. Resumen estadístico inicial de las variables numéricas:


,count,mean,min,25%,50%,75%,max,std
AÑO,309347.0,2020.262265,2015.0,2018.0,2021.0,2023.0,2026.0,3.126448
FECHA,309347,2020-09-24 17:00:21.843431680,2015-01-01 00:00:00,2018-01-01 00:00:00,2021-03-01 00:00:00,2023-01-01 00:00:00,2026-03-01 00:00:00,NaN
TOTAL,309347.0,8312906.773308,-129217034.0,17995.53502,312178.0,2157752.3095,5438516226.0,63086123.021323



3. Vista previa de las primeras 5 filas:


,AÑO,FECHA,COMPANIA,RAMOS,CIUDAD,SUCURSAL,REGIONAL,TOTAL
0,2015,2015-05-01,ACE,AUTOS,BARRANQUILLA,ATLANTICO,CARIBE,-594.0
1,2015,2015-06-01,ACE,AUTOS,BARRANQUILLA,ATLANTICO,CARIBE,-594.0
2,2015,2015-07-01,ACE,AUTOS,BARRANQUILLA,ATLANTICO,CARIBE,-594.0
3,2015,2015-08-01,ACE,AUTOS,BARRANQUILLA,ATLANTICO,CARIBE,-1703.0
4,2015,2015-09-01,ACE,AUTOS,BARRANQUILLA,ATLANTICO,CARIBE,-1703.0


In [13]:
# ==============================================================================
# BLOQUE 8.1: CREACIÓN DEL DATAFRAME DE CIERRE ANUAL (DICIEMBRE)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Filtramos estrictamente el mes 12 (la "foto final" del año)
df_anual = df_maestro[df_maestro['FECHA'].dt.month == 12].copy()

# Generar listas de opciones para los filtros de la interfaz
companias_unicas = sorted(df_anual['COMPANIA'].dropna().unique().tolist())
ramos_unicos = sorted(df_anual['RAMOS'].dropna().unique().tolist())
regionales_unicas = sorted(df_anual['REGIONAL'].dropna().unique().tolist())
ciudades_unicas = sorted(df_anual['CIUDAD'].dropna().unique().tolist())
sucursales_unicas = sorted(df_anual['SUCURSAL'].dropna().unique().tolist())
anios_disponibles = sorted(df_anual['AÑO'].dropna().unique().tolist())
ultimo_anio = anios_disponibles[-1] if anios_disponibles else None

print(f"✅ Se ha creado 'df_anual' con la foto de cierre de diciembre ({len(df_anual)} registros).")

✅ Se ha creado 'df_anual' con la foto de cierre de diciembre (23053 registros).


# Anual por ramo

In [20]:
# ==============================================================================
# BLOQUE 8.2: TABLERO ANUAL - RAMO
# ==============================================================================

# --- WIDGETS ---
dd_anio_ramo = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
sel_anios_atras_ramo = widgets.SelectMultiple(options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})
sel_cias_ramo = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Compañías:', layout={'width': '350px', 'height': '150px'})
sel_ramos_ramo = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '350px', 'height': '150px'})
btn_ramo = widgets.Button(description='Generar Tablero Ramo', button_style='primary', icon='list', layout={'width': '300px', 'height': '40px'})

ui_ramo = widgets.VBox([
    widgets.HBox([dd_anio_ramo, widgets.Label(" | "), sel_anios_atras_ramo]),
    widgets.HBox([sel_cias_ramo, sel_ramos_ramo]),
    btn_ramo
], layout={'border': '1px solid #ddd', 'padding': '10px'})
out_ramo = widgets.Output()

# --- MOTOR ANALÍTICO ---
def generar_tablero_ramo(b):
    with out_ramo:
        clear_output(wait=True)
        anio_base, anios_atras_lista = dd_anio_ramo.value, list(sel_anios_atras_ramo.value)
        companias_obj, ramo_obj = list(sel_cias_ramo.value), list(sel_ramos_ramo.value)

        if not companias_obj or not ramo_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones."))
            return

        display(Markdown(f"# 📊 Análisis Anual por Ramo"))
        display(Markdown("---"))

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"
            df_base = df_anual[(df_anual['AÑO'].isin([anio_base, anio_ant])) & (df_anual['RAMOS'].isin(ramo_obj))].copy()
            if df_base.empty: continue

            df_base['ETIQUETA_TIEMPO'] = np.where(df_base['AÑO'] == anio_base, lbl_actual, lbl_anterior)

            df_mkt = df_base.groupby(['ETIQUETA_TIEMPO', 'RAMOS'])['TOTAL'].sum().reset_index()
            df_mkt['COMPANIA'] = 'MERCADO'
            df_cias = df_base[df_base['COMPANIA'].isin(companias_obj)].groupby(['ETIQUETA_TIEMPO', 'COMPANIA', 'RAMOS'])['TOTAL'].sum().reset_index()

            df_pivot = pd.concat([df_cias, df_mkt]).pivot_table(index='RAMOS', columns=['COMPANIA', 'ETIQUETA_TIEMPO'], values='TOTAL', aggfunc='sum').fillna(0)
            if df_pivot.empty: continue
            df_pivot.loc['TOTAL'] = df_pivot.sum()

            frames, orden_col = [], ['MERCADO'] + companias_obj

            ayer_mkt = df_pivot.get(('MERCADO', lbl_anterior), pd.Series(0, index=df_pivot.index))
            hoy_mkt = df_pivot.get(('MERCADO', lbl_actual), pd.Series(0, index=df_pivot.index))
            var_mkt = pd.Series(np.nan, index=df_pivot.index)
            var_mkt[ayer_mkt != 0] = (hoy_mkt[ayer_mkt != 0] / ayer_mkt[ayer_mkt != 0]) - 1

            for cia in orden_col:
                if cia not in df_pivot.columns.get_level_values(0): continue
                ayer = df_pivot.get((cia, lbl_anterior), pd.Series(0, index=df_pivot.index))
                hoy = df_pivot.get((cia, lbl_actual), pd.Series(0, index=df_pivot.index))
                var = pd.Series(np.nan, index=df_pivot.index)
                var[ayer != 0] = (hoy[ayer != 0] / ayer[ayer != 0]) - 1
                veces = pd.Series(np.nan, index=df_pivot.index)
                m_v = var_mkt.notna() & (var_mkt != 0)
                veces[m_v] = var[m_v] / var_mkt[m_v]
                frames.append(pd.DataFrame({(cia, lbl_anterior): ayer, (cia, lbl_actual): hoy, (cia, 'Var'): var, (cia, 'Veces'): veces}))

            if not frames: continue
            tablero = pd.concat(frames, axis=1)

            for cia in orden_col:
                if (cia, lbl_anterior) in tablero.columns:
                    mask = (tablero[(cia, lbl_anterior)] == 0) & (tablero[(cia, lbl_actual)] == 0)
                    if mask.any():
                        for col in [(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var'), (cia, 'Veces')]:
                            tablero[col] = tablero[col].astype(object)
                        tablero.loc[mask, [(cia, lbl_anterior), (cia, lbl_actual)]] = "Sin registros"
                        tablero.loc[mask, [(cia, 'Var'), (cia, 'Veces')]] = "-"

            orden_filas = sorted([r for r in ramo_obj if r in tablero.index]) + (['TOTAL'] if 'TOTAL' in tablero.index else [])
            tablero = tablero.loc[orden_filas]

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))

            def fmt_m(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"${v:,.0f}")
            def fmt_p(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:+.2%}")
            def fmt_v(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:.2f}x")

            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in tablero.columns}

            def estilo_celdas(data):
                df_st = pd.DataFrame('', index=data.index, columns=data.columns)
                for cia in orden_col:
                    if (cia, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(cia, 'Var')], errors='coerce')
                        v_mkt = pd.to_numeric(data[('MERCADO', 'Var')], errors='coerce')
                        df_st[(cia, 'Var')] = np.where(v_cia > 0, 'color: #1E8449;', np.where(v_cia < 0, 'color: #E74C3C;', 'color: #7F8C8D;'))
                        df_st[(cia, 'Veces')] = np.where((v_cia - v_mkt) > 0, 'color: #1E8449; font-weight: bold;', np.where((v_cia - v_mkt) < 0, 'color: #E74C3C; font-weight: bold;', 'color: #7F8C8D;'))
                        m_sin = data[(cia, lbl_actual)] == "Sin registros"
                        if m_sin.any():
                            df_st.loc[m_sin, (cia, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                            df_st.loc[m_sin, (cia, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                if 'TOTAL' in df_st.index:
                    df_st.loc['TOTAL'] = 'background-color: #d1f2eb; font-weight: bold; border-top: 2px solid black; color: black;'
                return df_st

            display(tablero.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold')]},
                {'selector': 'td', 'props': [('text-align', 'right')]},
                {'selector': 'td:nth-child(4n+1)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

btn_ramo.on_click(generar_tablero_ramo)
display(ui_ramo, out_ramo)

Output()

# Por ciudad

In [21]:
# ==============================================================================
# BLOQUE 8.4: TABLERO ANUAL - CIUDAD
# ==============================================================================

# --- WIDGETS ---
dd_anio_ciu = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
sel_anios_atras_ciu = widgets.SelectMultiple(options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})
sel_cias_ciu = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Compañías:', layout={'width': '350px', 'height': '150px'})
sel_ciudades = widgets.SelectMultiple(options=ciudades_unicas, value=[], description='🏙️ Ciudades:', layout={'width': '350px', 'height': '150px'})
btn_ciu = widgets.Button(description='Generar Tablero Ciudad', button_style='primary', icon='building', layout={'width': '300px', 'height': '40px'})

ui_ciu = widgets.VBox([
    widgets.HBox([dd_anio_ciu, widgets.Label(" | "), sel_anios_atras_ciu]),
    widgets.HBox([sel_cias_ciu, sel_ciudades]),
    btn_ciu
], layout={'border': '1px solid #ddd', 'padding': '10px'})
out_ciu = widgets.Output()

# --- MOTOR ANALÍTICO ---
def generar_tablero_ciudad(b):
    with out_ciu:
        clear_output(wait=True)
        anio_base, anios_atras_lista = dd_anio_ciu.value, list(sel_anios_atras_ciu.value)
        companias_obj, ciu_obj = list(sel_cias_ciu.value), list(sel_ciudades.value)

        if not companias_obj or not ciu_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones."))
            return

        display(Markdown(f"# 📊 Análisis Anual por Ciudad"))
        display(Markdown("---"))

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"
            df_base = df_anual[(df_anual['AÑO'].isin([anio_base, anio_ant])) & (df_anual['CIUDAD'].isin(ciu_obj))].copy()
            if df_base.empty: continue

            df_base['ETIQUETA_TIEMPO'] = np.where(df_base['AÑO'] == anio_base, lbl_actual, lbl_anterior)

            df_mkt = df_base.groupby(['ETIQUETA_TIEMPO', 'CIUDAD'])['TOTAL'].sum().reset_index()
            df_mkt['COMPANIA'] = 'MERCADO'
            df_cias = df_base[df_base['COMPANIA'].isin(companias_obj)].groupby(['ETIQUETA_TIEMPO', 'COMPANIA', 'CIUDAD'])['TOTAL'].sum().reset_index()

            df_pivot = pd.concat([df_cias, df_mkt]).pivot_table(index='CIUDAD', columns=['COMPANIA', 'ETIQUETA_TIEMPO'], values='TOTAL', aggfunc='sum').fillna(0)
            if df_pivot.empty: continue
            df_pivot.loc['TOTAL'] = df_pivot.sum()

            frames, orden_col = [], ['MERCADO'] + companias_obj

            ayer_mkt = df_pivot.get(('MERCADO', lbl_anterior), pd.Series(0, index=df_pivot.index))
            hoy_mkt = df_pivot.get(('MERCADO', lbl_actual), pd.Series(0, index=df_pivot.index))
            var_mkt = pd.Series(np.nan, index=df_pivot.index)
            var_mkt[ayer_mkt != 0] = (hoy_mkt[ayer_mkt != 0] / ayer_mkt[ayer_mkt != 0]) - 1

            for cia in orden_col:
                if cia not in df_pivot.columns.get_level_values(0): continue
                ayer = df_pivot.get((cia, lbl_anterior), pd.Series(0, index=df_pivot.index))
                hoy = df_pivot.get((cia, lbl_actual), pd.Series(0, index=df_pivot.index))
                var = pd.Series(np.nan, index=df_pivot.index)
                var[ayer != 0] = (hoy[ayer != 0] / ayer[ayer != 0]) - 1
                veces = pd.Series(np.nan, index=df_pivot.index)
                m_v = var_mkt.notna() & (var_mkt != 0)
                veces[m_v] = var[m_v] / var_mkt[m_v]
                frames.append(pd.DataFrame({(cia, lbl_anterior): ayer, (cia, lbl_actual): hoy, (cia, 'Var'): var, (cia, 'Veces'): veces}))

            if not frames: continue
            tablero = pd.concat(frames, axis=1)

            for cia in orden_col:
                if (cia, lbl_anterior) in tablero.columns:
                    mask = (tablero[(cia, lbl_anterior)] == 0) & (tablero[(cia, lbl_actual)] == 0)
                    if mask.any():
                        for col in [(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var'), (cia, 'Veces')]:
                            tablero[col] = tablero[col].astype(object)
                        tablero.loc[mask, [(cia, lbl_anterior), (cia, lbl_actual)]] = "Sin registros"
                        tablero.loc[mask, [(cia, 'Var'), (cia, 'Veces')]] = "-"

            orden_filas = sorted([r for r in ciu_obj if r in tablero.index]) + (['TOTAL'] if 'TOTAL' in tablero.index else [])
            tablero = tablero.loc[orden_filas]

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))

            def fmt_m(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"${v:,.0f}")
            def fmt_p(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:+.2%}")
            def fmt_v(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:.2f}x")

            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in tablero.columns}

            def estilo_celdas(data):
                df_st = pd.DataFrame('', index=data.index, columns=data.columns)
                for cia in orden_col:
                    if (cia, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(cia, 'Var')], errors='coerce')
                        v_mkt = pd.to_numeric(data[('MERCADO', 'Var')], errors='coerce')
                        df_st[(cia, 'Var')] = np.where(v_cia > 0, 'color: #1E8449;', np.where(v_cia < 0, 'color: #E74C3C;', 'color: #7F8C8D;'))
                        df_st[(cia, 'Veces')] = np.where((v_cia - v_mkt) > 0, 'color: #1E8449; font-weight: bold;', np.where((v_cia - v_mkt) < 0, 'color: #E74C3C; font-weight: bold;', 'color: #7F8C8D;'))
                        m_sin = data[(cia, lbl_actual)] == "Sin registros"
                        if m_sin.any():
                            df_st.loc[m_sin, (cia, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                            df_st.loc[m_sin, (cia, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                if 'TOTAL' in df_st.index:
                    df_st.loc['TOTAL'] = 'background-color: #d1f2eb; font-weight: bold; border-top: 2px solid black; color: black;'
                return df_st

            display(tablero.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold')]},
                {'selector': 'td', 'props': [('text-align', 'right')]},
                {'selector': 'td:nth-child(4n+1)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

btn_ciu.on_click(generar_tablero_ciudad)
display(ui_ciu, out_ciu)

Output()

# Por sucursal

In [23]:
# ==============================================================================
# BLOQUE 8.5: TABLERO ANUAL - SUCURSAL
# ==============================================================================

# --- WIDGETS ---
dd_anio_suc = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
sel_anios_atras_suc = widgets.SelectMultiple(options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})
sel_cias_suc = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Compañías:', layout={'width': '350px', 'height': '150px'})
sel_sucursales = widgets.SelectMultiple(options=sucursales_unicas, value=[], description='📍 Sucursales:', layout={'width': '350px', 'height': '150px'})
btn_suc = widgets.Button(description='Generar Tablero Sucursal', button_style='primary', icon='sitemap', layout={'width': '300px', 'height': '40px'})

ui_suc = widgets.VBox([
    widgets.HBox([dd_anio_suc, widgets.Label(" | "), sel_anios_atras_suc]),
    widgets.HBox([sel_cias_suc, sel_sucursales]),
    btn_suc
], layout={'border': '1px solid #ddd', 'padding': '10px'})
out_suc = widgets.Output()

# --- MOTOR ANALÍTICO ---
def generar_tablero_sucursal(b):
    with out_suc:
        clear_output(wait=True)
        anio_base, anios_atras_lista = dd_anio_suc.value, list(sel_anios_atras_suc.value)
        companias_obj, suc_obj = list(sel_cias_suc.value), list(sel_sucursales.value)

        if not companias_obj or not suc_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones."))
            return

        display(Markdown(f"# 📊 Análisis Anual por Sucursal"))
        display(Markdown("---"))

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"
            df_base = df_anual[(df_anual['AÑO'].isin([anio_base, anio_ant])) & (df_anual['SUCURSAL'].isin(suc_obj))].copy()
            if df_base.empty: continue

            df_base['ETIQUETA_TIEMPO'] = np.where(df_base['AÑO'] == anio_base, lbl_actual, lbl_anterior)

            df_mkt = df_base.groupby(['ETIQUETA_TIEMPO', 'SUCURSAL'])['TOTAL'].sum().reset_index()
            df_mkt['COMPANIA'] = 'MERCADO'
            df_cias = df_base[df_base['COMPANIA'].isin(companias_obj)].groupby(['ETIQUETA_TIEMPO', 'COMPANIA', 'SUCURSAL'])['TOTAL'].sum().reset_index()

            df_pivot = pd.concat([df_cias, df_mkt]).pivot_table(index='SUCURSAL', columns=['COMPANIA', 'ETIQUETA_TIEMPO'], values='TOTAL', aggfunc='sum').fillna(0)
            if df_pivot.empty: continue
            df_pivot.loc['TOTAL'] = df_pivot.sum()

            frames, orden_col = [], ['MERCADO'] + companias_obj

            ayer_mkt = df_pivot.get(('MERCADO', lbl_anterior), pd.Series(0, index=df_pivot.index))
            hoy_mkt = df_pivot.get(('MERCADO', lbl_actual), pd.Series(0, index=df_pivot.index))
            var_mkt = pd.Series(np.nan, index=df_pivot.index)
            var_mkt[ayer_mkt != 0] = (hoy_mkt[ayer_mkt != 0] / ayer_mkt[ayer_mkt != 0]) - 1

            for cia in orden_col:
                if cia not in df_pivot.columns.get_level_values(0): continue
                ayer = df_pivot.get((cia, lbl_anterior), pd.Series(0, index=df_pivot.index))
                hoy = df_pivot.get((cia, lbl_actual), pd.Series(0, index=df_pivot.index))
                var = pd.Series(np.nan, index=df_pivot.index)
                var[ayer != 0] = (hoy[ayer != 0] / ayer[ayer != 0]) - 1
                veces = pd.Series(np.nan, index=df_pivot.index)
                m_v = var_mkt.notna() & (var_mkt != 0)
                veces[m_v] = var[m_v] / var_mkt[m_v]
                frames.append(pd.DataFrame({(cia, lbl_anterior): ayer, (cia, lbl_actual): hoy, (cia, 'Var'): var, (cia, 'Veces'): veces}))

            if not frames: continue
            tablero = pd.concat(frames, axis=1)

            for cia in orden_col:
                if (cia, lbl_anterior) in tablero.columns:
                    mask = (tablero[(cia, lbl_anterior)] == 0) & (tablero[(cia, lbl_actual)] == 0)
                    if mask.any():
                        for col in [(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var'), (cia, 'Veces')]:
                            tablero[col] = tablero[col].astype(object)
                        tablero.loc[mask, [(cia, lbl_anterior), (cia, lbl_actual)]] = "Sin registros"
                        tablero.loc[mask, [(cia, 'Var'), (cia, 'Veces')]] = "-"

            orden_filas = sorted([r for r in suc_obj if r in tablero.index]) + (['TOTAL'] if 'TOTAL' in tablero.index else [])
            tablero = tablero.loc[orden_filas]

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))

            def fmt_m(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"${v:,.0f}")
            def fmt_p(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:+.2%}")
            def fmt_v(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:.2f}x")

            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in tablero.columns}

            def estilo_celdas(data):
                df_st = pd.DataFrame('', index=data.index, columns=data.columns)
                for cia in orden_col:
                    if (cia, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(cia, 'Var')], errors='coerce')
                        v_mkt = pd.to_numeric(data[('MERCADO', 'Var')], errors='coerce')
                        df_st[(cia, 'Var')] = np.where(v_cia > 0, 'color: #1E8449;', np.where(v_cia < 0, 'color: #E74C3C;', 'color: #7F8C8D;'))
                        df_st[(cia, 'Veces')] = np.where((v_cia - v_mkt) > 0, 'color: #1E8449; font-weight: bold;', np.where((v_cia - v_mkt) < 0, 'color: #E74C3C; font-weight: bold;', 'color: #7F8C8D;'))
                        m_sin = data[(cia, lbl_actual)] == "Sin registros"
                        if m_sin.any():
                            df_st.loc[m_sin, (cia, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                            df_st.loc[m_sin, (cia, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                if 'TOTAL' in df_st.index:
                    df_st.loc['TOTAL'] = 'background-color: #d1f2eb; font-weight: bold; border-top: 2px solid black; color: black;'
                return df_st

            display(tablero.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold')]},
                {'selector': 'td', 'props': [('text-align', 'right')]},
                {'selector': 'td:nth-child(4n+1)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

btn_suc.on_click(generar_tablero_sucursal)
display(ui_suc, out_suc)

Output()

# Por regional

In [24]:
# ==============================================================================
# BLOQUE 8.3: TABLERO ANUAL - REGIONAL
# ==============================================================================

# --- WIDGETS ---
dd_anio_reg = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='📅 Año Base:', layout={'width': '200px'})
sel_anios_atras_reg = widgets.SelectMultiple(options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})
sel_cias_reg = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Compañías:', layout={'width': '350px', 'height': '150px'})
sel_regionales = widgets.SelectMultiple(options=regionales_unicas, value=[], description='🗺️ Regionales:', layout={'width': '350px', 'height': '150px'})
btn_reg = widgets.Button(description='Generar Tablero Regional', button_style='primary', icon='map', layout={'width': '300px', 'height': '40px'})

ui_reg = widgets.VBox([
    widgets.HBox([dd_anio_reg, widgets.Label(" | "), sel_anios_atras_reg]),
    widgets.HBox([sel_cias_reg, sel_regionales]),
    btn_reg
], layout={'border': '1px solid #ddd', 'padding': '10px'})
out_reg = widgets.Output()

# --- MOTOR ANALÍTICO ---
def generar_tablero_regional(b):
    with out_reg:
        clear_output(wait=True)
        anio_base, anios_atras_lista = dd_anio_reg.value, list(sel_anios_atras_reg.value)
        companias_obj, reg_obj = list(sel_cias_reg.value), list(sel_regionales.value)

        if not companias_obj or not reg_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones."))
            return

        display(Markdown(f"# 📊 Análisis Anual por Regional"))
        display(Markdown("---"))

        for anios_salto in sorted(anios_atras_lista):
            anio_ant = anio_base - anios_salto
            if anio_ant not in anios_disponibles: continue

            lbl_actual, lbl_anterior = f"Cierre ({anio_base})", f"Anterior ({anio_ant})"
            df_base = df_anual[(df_anual['AÑO'].isin([anio_base, anio_ant])) & (df_anual['REGIONAL'].isin(reg_obj))].copy()
            if df_base.empty: continue

            df_base['ETIQUETA_TIEMPO'] = np.where(df_base['AÑO'] == anio_base, lbl_actual, lbl_anterior)

            df_mkt = df_base.groupby(['ETIQUETA_TIEMPO', 'REGIONAL'])['TOTAL'].sum().reset_index()
            df_mkt['COMPANIA'] = 'MERCADO'
            df_cias = df_base[df_base['COMPANIA'].isin(companias_obj)].groupby(['ETIQUETA_TIEMPO', 'COMPANIA', 'REGIONAL'])['TOTAL'].sum().reset_index()

            df_pivot = pd.concat([df_cias, df_mkt]).pivot_table(index='REGIONAL', columns=['COMPANIA', 'ETIQUETA_TIEMPO'], values='TOTAL', aggfunc='sum').fillna(0)
            if df_pivot.empty: continue
            df_pivot.loc['TOTAL'] = df_pivot.sum()

            frames, orden_col = [], ['MERCADO'] + companias_obj

            ayer_mkt = df_pivot.get(('MERCADO', lbl_anterior), pd.Series(0, index=df_pivot.index))
            hoy_mkt = df_pivot.get(('MERCADO', lbl_actual), pd.Series(0, index=df_pivot.index))
            var_mkt = pd.Series(np.nan, index=df_pivot.index)
            var_mkt[ayer_mkt != 0] = (hoy_mkt[ayer_mkt != 0] / ayer_mkt[ayer_mkt != 0]) - 1

            for cia in orden_col:
                if cia not in df_pivot.columns.get_level_values(0): continue
                ayer = df_pivot.get((cia, lbl_anterior), pd.Series(0, index=df_pivot.index))
                hoy = df_pivot.get((cia, lbl_actual), pd.Series(0, index=df_pivot.index))
                var = pd.Series(np.nan, index=df_pivot.index)
                var[ayer != 0] = (hoy[ayer != 0] / ayer[ayer != 0]) - 1
                veces = pd.Series(np.nan, index=df_pivot.index)
                m_v = var_mkt.notna() & (var_mkt != 0)
                veces[m_v] = var[m_v] / var_mkt[m_v]
                frames.append(pd.DataFrame({(cia, lbl_anterior): ayer, (cia, lbl_actual): hoy, (cia, 'Var'): var, (cia, 'Veces'): veces}))

            if not frames: continue
            tablero = pd.concat(frames, axis=1)

            for cia in orden_col:
                if (cia, lbl_anterior) in tablero.columns:
                    mask = (tablero[(cia, lbl_anterior)] == 0) & (tablero[(cia, lbl_actual)] == 0)
                    if mask.any():
                        for col in [(cia, lbl_anterior), (cia, lbl_actual), (cia, 'Var'), (cia, 'Veces')]:
                            tablero[col] = tablero[col].astype(object)
                        tablero.loc[mask, [(cia, lbl_anterior), (cia, lbl_actual)]] = "Sin registros"
                        tablero.loc[mask, [(cia, 'Var'), (cia, 'Veces')]] = "-"

            orden_filas = sorted([r for r in reg_obj if r in tablero.index]) + (['TOTAL'] if 'TOTAL' in tablero.index else [])
            tablero = tablero.loc[orden_filas]

            display(Markdown(f"### ➡️ Comparativa: **{anio_ant}** vs **{anio_base}**"))

            def fmt_m(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"${v:,.0f}")
            def fmt_p(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:+.2%}")
            def fmt_v(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:.2f}x")

            fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in tablero.columns}

            def estilo_celdas(data):
                df_st = pd.DataFrame('', index=data.index, columns=data.columns)
                for cia in orden_col:
                    if (cia, 'Var') in data.columns:
                        v_cia = pd.to_numeric(data[(cia, 'Var')], errors='coerce')
                        v_mkt = pd.to_numeric(data[('MERCADO', 'Var')], errors='coerce')
                        df_st[(cia, 'Var')] = np.where(v_cia > 0, 'color: #1E8449;', np.where(v_cia < 0, 'color: #E74C3C;', 'color: #7F8C8D;'))
                        df_st[(cia, 'Veces')] = np.where((v_cia - v_mkt) > 0, 'color: #1E8449; font-weight: bold;', np.where((v_cia - v_mkt) < 0, 'color: #E74C3C; font-weight: bold;', 'color: #7F8C8D;'))
                        m_sin = data[(cia, lbl_actual)] == "Sin registros"
                        if m_sin.any():
                            df_st.loc[m_sin, (cia, lbl_anterior)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                            df_st.loc[m_sin, (cia, lbl_actual)] = 'color: #bdc3c7; font-style: italic; font-size: 11px; text-align: center;'
                if 'TOTAL' in df_st.index:
                    df_st.loc['TOTAL'] = 'background-color: #d1f2eb; font-weight: bold; border-top: 2px solid black; color: black;'
                return df_st

            display(tablero.style.format(fc, na_rep="-").apply(estilo_celdas, axis=None).set_table_styles([
                {'selector': 'th.col_heading.level0', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center')]},
                {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center')]},
                {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold')]},
                {'selector': 'td', 'props': [('text-align', 'right')]},
                {'selector': 'td:nth-child(4n+1)', 'props': [('border-right', '2px solid #bdc3c7')]}
            ]))

btn_reg.on_click(generar_tablero_regional)
display(ui_reg, out_reg)

Output()

In [25]:
# ==============================================================================
# BLOQUE 16: DASHBOARD MAESTRO TIPO POWER BI (RESUMEN EJECUTIVO ACTUALIZADO)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output, HTML
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')

# 1. Extracción de Parámetros
# ------------------------------------------------------------------------------
companias_unicas = sorted(df_maestro['COMPANIA'].dropna().unique().tolist())
ramos_unicos = sorted(df_maestro['RAMOS'].dropna().unique().tolist())
regionales_unicas = sorted(df_maestro['REGIONAL'].dropna().unique().tolist())
fechas_disponibles = sorted(df_maestro['FECHA'].dropna().unique())
anios_disponibles = sorted(df_maestro['AÑO'].dropna().unique().tolist())

meses_dict = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
              7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
meses_inversos = {v: k for k, v in meses_dict.items()}
meses_nombres = list(meses_dict.values())

COLOR_VERDE = '#0F753B'
COLOR_CREMA = '#FCE49C'

# 2. Construcción de UI
# ------------------------------------------------------------------------------
ultimo_anio = fechas_disponibles[-1].year
ultimo_mes_txt = meses_dict[fechas_disponibles[-1].month]

dropdown_anio_inicio = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='Inicio - Año:', layout={'width': '160px'})
dropdown_mes_inicio = widgets.Dropdown(options=meses_nombres, value=meses_dict[1], description='Mes:', layout={'width': '140px'})
dropdown_anio_fin = widgets.Dropdown(options=anios_disponibles, value=ultimo_anio, description='Final - Año:', layout={'width': '160px'})
dropdown_mes_fin = widgets.Dropdown(options=meses_nombres, value=ultimo_mes_txt, description='Mes:', layout={'width': '140px'})
input_meses = widgets.BoundedIntText(value=1, min=1, max=1000, description='Cant. Meses:', layout={'width': '150px'})
opciones_anios_atras = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
select_anios_atras = widgets.SelectMultiple(options=opciones_anios_atras, value=[1], description='⏳ Años Atrás:', layout={'width': '150px', 'height': '80px'})
mensaje_alerta = widgets.HTML(value="")

def actualizar_meses_desde_fechas(*args):
    input_meses.unobserve(actualizar_fechas_desde_meses, 'value')
    fecha_ini = pd.Timestamp(year=dropdown_anio_inicio.value, month=meses_inversos[dropdown_mes_inicio.value], day=1)
    fecha_fin = pd.Timestamp(year=dropdown_anio_fin.value, month=meses_inversos[dropdown_mes_fin.value], day=1)
    if fecha_fin >= fecha_ini:
        diff_meses = (fecha_fin.year - fecha_ini.year) * 12 + (fecha_fin.month - fecha_ini.month) + 1
        input_meses.value = diff_meses
        anios_salto_max = max(select_anios_atras.value) if select_anios_atras.value else 0
        fecha_ini_ant_max = fecha_ini - relativedelta(years=anios_salto_max)
        if anios_salto_max > 0 and fecha_ini_ant_max < fechas_disponibles[0]:
            mensaje_alerta.value = f"<span style='color:#E74C3C; font-weight:bold;'>⚠️ Alerta: No hay historia para comparar {anios_salto_max} año(s) atrás.</span>"
        else:
            mensaje_alerta.value = f"<span style='color:#1E8449; font-weight:bold;'>✅ Periodo válido.</span>"
    else: mensaje_alerta.value = ""
    input_meses.observe(actualizar_fechas_desde_meses, 'value')

def actualizar_fechas_desde_meses(*args):
    dropdown_anio_fin.unobserve(actualizar_meses_desde_fechas, 'value')
    dropdown_mes_fin.unobserve(actualizar_meses_desde_fechas, 'value')
    fecha_ini = pd.Timestamp(year=dropdown_anio_inicio.value, month=meses_inversos[dropdown_mes_inicio.value], day=1)
    nueva_fecha_fin = fecha_ini + relativedelta(months=input_meses.value - 1)
    if nueva_fecha_fin > fechas_disponibles[-1]: nueva_fecha_fin = fechas_disponibles[-1]
    dropdown_anio_fin.value = nueva_fecha_fin.year
    dropdown_mes_fin.value = meses_dict[nueva_fecha_fin.month]
    dropdown_anio_fin.observe(actualizar_meses_desde_fechas, 'value')
    dropdown_mes_fin.observe(actualizar_meses_desde_fechas, 'value')
    actualizar_meses_desde_fechas()

dropdown_anio_inicio.observe(actualizar_meses_desde_fechas, 'value')
dropdown_mes_inicio.observe(actualizar_meses_desde_fechas, 'value')
dropdown_anio_fin.observe(actualizar_meses_desde_fechas, 'value')
dropdown_mes_fin.observe(actualizar_meses_desde_fechas, 'value')
input_meses.observe(actualizar_fechas_desde_meses, 'value')
select_anios_atras.observe(actualizar_meses_desde_fechas, 'value')
actualizar_meses_desde_fechas()

select_ramos = widgets.SelectMultiple(options=ramos_unicos, value=[], description='📋 Ramos:', layout={'width': '250px', 'height': '120px'})
select_companias = widgets.SelectMultiple(options=companias_unicas, value=[], description='🏢 Cías:', layout={'width': '250px', 'height': '120px'})
select_regionales = widgets.SelectMultiple(options=regionales_unicas, value=[], description='🗺️ Reg:', layout={'width': '250px', 'height': '120px'})

boton_generar = widgets.Button(description='Generar Dashboard', button_style='success', icon='tachometer', layout={'width': '300px', 'height': '40px'})
out_main = widgets.Output()

# 3. Funciones de Apoyo Matemático y Formato
# ------------------------------------------------------------------------------
def get_pivot(df_base, dimension, entidades, lbl_actual, lbl_anterior):
    df_m = df_base.groupby(['ETIQUETA_TIEMPO', dimension])['TOTAL'].sum().reset_index()
    df_m['COMPANIA'] = 'MERCADO'
    df_c = df_base[df_base['COMPANIA'].isin(entidades)].groupby(['ETIQUETA_TIEMPO', 'COMPANIA', dimension])['TOTAL'].sum().reset_index()
    df_con = pd.concat([df_c, df_m], ignore_index=True)
    df_tot = df_con.groupby(['ETIQUETA_TIEMPO', 'COMPANIA'])['TOTAL'].sum().reset_index()
    df_tot[dimension] = 'TOTAL'
    df_fin = pd.concat([df_con, df_tot], ignore_index=True)
    df_piv = df_fin.pivot_table(index=dimension, columns=['COMPANIA', 'ETIQUETA_TIEMPO'], values='TOTAL', aggfunc='sum').fillna(0)

    frames = []
    ord_c = ['MERCADO'] + [c for c in entidades]

    a_mkt = df_piv.get(('MERCADO', lbl_anterior), pd.Series(0, index=df_piv.index))
    h_mkt = df_piv.get(('MERCADO', lbl_actual), pd.Series(0, index=df_piv.index))
    v_mkt = pd.Series(np.nan, index=df_piv.index)
    v_mkt[a_mkt != 0] = (h_mkt[a_mkt != 0] / a_mkt[a_mkt != 0]) - 1

    for cia in ord_c:
        if cia not in df_piv.columns.get_level_values(0): continue
        ayer = df_piv.get((cia, lbl_anterior), pd.Series(0, index=df_piv.index))
        hoy = df_piv.get((cia, lbl_actual), pd.Series(0, index=df_piv.index))
        var = pd.Series(np.nan, index=df_piv.index)
        var[ayer != 0] = (hoy[ayer != 0] / ayer[ayer != 0]) - 1
        veces = pd.Series(np.nan, index=df_piv.index)
        m_v = v_mkt.notna() & (v_mkt != 0)
        veces[m_v] = var[m_v] / v_mkt[m_v]
        frames.append(pd.DataFrame({(cia, lbl_anterior): ayer, (cia, lbl_actual): hoy, (cia, 'Var'): var, (cia, 'Veces'): veces}))

    if not frames: return pd.DataFrame(), []
    tab = pd.concat(frames, axis=1)
    return tab, ord_c

def fmt_m(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"${v:,.0f}")
def fmt_p(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:+.2%}")
def fmt_v(v): return "-" if pd.isna(v) else (v if isinstance(v, str) else f"{v:.2f}x")

def aplicar_estilos(data, ord_c, lbl_a, lbl_h):
    df_e = pd.DataFrame('', index=data.index, columns=data.columns)
    for c in ord_c:
        if (c, 'Var') in data.columns and (c, 'Veces') in data.columns:
            vc, vm = pd.to_numeric(data[(c, 'Var')], errors='coerce'), pd.to_numeric(data[('MERCADO', 'Var')], errors='coerce')
            df_e[(c, 'Var')] = np.where(vc > 0, 'color: #1E8449;', np.where(vc < 0, 'color: #E74C3C;', 'color: #7F8C8D;'))
            df_e[(c, 'Veces')] = np.where((vc-vm) > 0, 'color: #1E8449; font-weight: bold;', np.where((vc-vm) < 0, 'color: #E74C3C; font-weight: bold;', 'color: #7F8C8D;'))
    if 'TOTAL' in df_e.index:
        df_e.loc['TOTAL'] = f'background-color: #ecf0f1; font-weight: bold; border-top: 2px solid {COLOR_VERDE};'
    return df_e

tbl_styles = [
    {'selector': 'th.col_heading.level0', 'props': [(f'background-color', COLOR_VERDE), ('color', 'white'), ('text-align', 'center'), ('font-size', '11px'), ('border-right', '1px solid white')]},
    {'selector': 'th.col_heading.level1', 'props': [('background-color', '#ecf0f1'), ('color', 'black'), ('text-align', 'center'), ('font-size', '10px')]},
    {'selector': 'th.row_heading', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('font-size', '10px')]},
    {'selector': 'td', 'props': [('text-align', 'right'), ('font-size', '11px')]},
    {'selector': 'td:nth-child(4n)', 'props': [('border-right', '2px solid #bdc3c7')]}
]

def header_html(titulo):
    return f"<div style='background-color:{COLOR_CREMA}; padding:4px; text-align:center; border:1px solid {COLOR_VERDE}; margin-bottom: 5px;'><h4 style='margin:0; color:#333; font-size:12px; font-weight:bold;'>{titulo}</h4></div>"

# 4. Motor Principal (Dashboard Layout)
# ------------------------------------------------------------------------------
def generar_tablero_maestro(b):
    with out_main:
        clear_output(wait=True)

        anio_inicio, mes_inicio = dropdown_anio_inicio.value, dropdown_mes_inicio.value
        anio_fin, mes_fin = dropdown_anio_fin.value, dropdown_mes_fin.value
        anios_atras_lista = list(select_anios_atras.value)
        ramos_obj = list(select_ramos.value)
        cias_obj = list(select_companias.value)
        reg_obj = list(select_regionales.value)

        if not ramos_obj or not cias_obj or not reg_obj or not anios_atras_lista:
            display(Markdown("⚠️ **Error:** Faltan selecciones."))
            return

        fecha_inicio = pd.Timestamp(year=anio_inicio, month=meses_inversos[mes_inicio], day=1)
        fecha_fin = pd.Timestamp(year=anio_fin, month=meses_inversos[mes_fin], day=1)
        fechas_periodo_actual = [f for f in fechas_disponibles if fecha_inicio <= f <= fecha_fin]

        for anios_salto in sorted(anios_atras_lista):
            fecha_inicio_ant = fecha_inicio - relativedelta(years=anios_salto)
            fecha_fin_ant = fecha_fin - relativedelta(years=anios_salto)
            if fecha_inicio_ant < fechas_disponibles[0]: continue

            fechas_anterior = [f for f in fechas_disponibles if fecha_inicio_ant <= f <= fecha_fin_ant]
            lbl_actual, lbl_anterior = f"Periodo ({fecha_fin.year})", f"Anterior ({fecha_fin_ant.year})"

            df_base = df_maestro[(df_maestro['FECHA'].isin(fechas_periodo_actual + fechas_anterior)) &
                                 (df_maestro['RAMOS'].isin(ramos_obj))].copy()

            df_base_reg = df_base[df_base['REGIONAL'].isin(reg_obj)].copy()
            if df_base_reg.empty: continue

            df_base['ETIQUETA_TIEMPO'] = np.where(df_base['FECHA'].isin(fechas_periodo_actual), lbl_actual, lbl_anterior)
            df_base_reg['ETIQUETA_TIEMPO'] = np.where(df_base_reg['FECHA'].isin(fechas_periodo_actual), lbl_actual, lbl_anterior)

            # CÁLCULOS ESTÁNDAR
            tab_reg, cols_r = get_pivot(df_base_reg, 'REGIONAL', cias_obj, lbl_actual, lbl_anterior)
            tab_ramo, cols_rm = get_pivot(df_base_reg, 'RAMOS', cias_obj, lbl_actual, lbl_anterior)

            # CONFIGURACIÓN DEL GRID
            box_layout = widgets.Layout(border=f'1px solid {COLOR_VERDE}', padding='5px', width='50%', height='350px', overflow='auto')
            box_full = widgets.Layout(border=f'1px solid {COLOR_VERDE}', padding='5px', width='100%', height='350px', overflow='auto')

            out_hier = widgets.Output(layout=box_layout) # Top Left (Jerarquía Año/Reg/Suc)
            out_ml = widgets.Output(layout=box_layout)   # Top Right (Tabla Reg)
            out_mr = widgets.Output(layout=box_layout)   # Mid Left (Gráfica Ramo)
            out_bl = widgets.Output(layout=box_layout)   # Mid Right (Tabla Ramo)
            out_br = widgets.Output(layout=box_full)     # Bot (Ranking Completo)

            # --- TOP LEFT: Tabla Jerárquica (Año -> Regional -> Sucursal) ---
            with out_hier:
                display(HTML(header_html("DETALLE AÑO, REGIONAL Y SUCURSAL")))

                # Pre-cálculos para jerarquía
                mkt_tot = df_base_reg.groupby('ETIQUETA_TIEMPO')['TOTAL'].sum()
                mkt_reg = df_base_reg.groupby(['REGIONAL', 'ETIQUETA_TIEMPO'])['TOTAL'].sum().unstack(fill_value=0)
                mkt_suc = df_base_reg.groupby(['REGIONAL', 'SUCURSAL', 'ETIQUETA_TIEMPO'])['TOTAL'].sum().unstack(fill_value=0)

                rows_hier = []
                # Fila 1: Año Total
                row_anio = {'Nivel': 0, 'Agrupación': f'Año {fecha_fin.year}'}
                m_ant_tot = mkt_tot.get(lbl_anterior, 0)
                m_act_tot = mkt_tot.get(lbl_actual, 0)
                var_m_tot = (m_act_tot / m_ant_tot - 1) if m_ant_tot != 0 else np.nan
                row_anio['Var Mercado'] = var_m_tot

                for cia in cias_obj:
                    df_cia_t = df_base_reg[df_base_reg['COMPANIA'] == cia]
                    cia_t = df_cia_t.groupby('ETIQUETA_TIEMPO')['TOTAL'].sum()
                    c_ant_tot = cia_t.get(lbl_anterior, 0)
                    c_act_tot = cia_t.get(lbl_actual, 0)
                    var_c_tot = (c_act_tot / c_ant_tot - 1) if c_ant_tot != 0 else np.nan
                    row_anio[f'Var {cia}'] = var_c_tot
                    row_anio[f'Veces {cia}'] = (var_c_tot / var_m_tot) if (pd.notna(var_m_tot) and var_m_tot != 0) else np.nan
                rows_hier.append(row_anio)

                # Filas: Regional -> Sucursales
                for reg in sorted(reg_obj):
                    if reg not in mkt_reg.index: continue
                    # Fila Regional
                    row_r = {'Nivel': 1, 'Agrupación': str(reg)}
                    m_ant_r = mkt_reg.loc[reg, lbl_anterior] if lbl_anterior in mkt_reg.columns else 0
                    m_act_r = mkt_reg.loc[reg, lbl_actual] if lbl_actual in mkt_reg.columns else 0
                    var_m_r = (m_act_r / m_ant_r - 1) if m_ant_r != 0 else np.nan
                    row_r['Var Mercado'] = var_m_r

                    for cia in cias_obj:
                        df_cia_r = df_base_reg[(df_base_reg['COMPANIA'] == cia) & (df_base_reg['REGIONAL'] == reg)]
                        cia_t_r = df_cia_r.groupby('ETIQUETA_TIEMPO')['TOTAL'].sum()
                        c_ant_r = cia_t_r.get(lbl_anterior, 0)
                        c_act_r = cia_t_r.get(lbl_actual, 0)
                        var_c_r = (c_act_r / c_ant_r - 1) if c_ant_r != 0 else np.nan
                        row_r[f'Var {cia}'] = var_c_r
                        row_r[f'Veces {cia}'] = (var_c_r / var_m_r) if (pd.notna(var_m_r) and var_m_r != 0) else np.nan
                    rows_hier.append(row_r)

                    # Filas Sucursal
                    sucursales = df_base_reg[df_base_reg['REGIONAL'] == reg]['SUCURSAL'].dropna().unique()
                    for suc in sorted(sucursales):
                        if (reg, suc) not in mkt_suc.index: continue
                        row_s = {'Nivel': 2, 'Agrupación': f"   ↳ {suc}"}
                        m_ant_s = mkt_suc.loc[(reg, suc), lbl_anterior] if lbl_anterior in mkt_suc.columns else 0
                        m_act_s = mkt_suc.loc[(reg, suc), lbl_actual] if lbl_actual in mkt_suc.columns else 0
                        var_m_s = (m_act_s / m_ant_s - 1) if m_ant_s != 0 else np.nan
                        row_s['Var Mercado'] = var_m_s

                        for cia in cias_obj:
                            df_cia_s = df_base_reg[(df_base_reg['COMPANIA'] == cia) & (df_base_reg['REGIONAL'] == reg) & (df_base_reg['SUCURSAL'] == suc)]
                            cia_t_s = df_cia_s.groupby('ETIQUETA_TIEMPO')['TOTAL'].sum()
                            c_ant_s = cia_t_s.get(lbl_anterior, 0)
                            c_act_s = cia_t_s.get(lbl_actual, 0)
                            var_c_s = (c_act_s / c_ant_s - 1) if c_ant_s != 0 else np.nan
                            row_s[f'Var {cia}'] = var_c_s
                            row_s[f'Veces {cia}'] = (var_c_s / var_m_s) if (pd.notna(var_m_s) and var_m_s != 0) else np.nan
                        rows_hier.append(row_s)

                df_hier = pd.DataFrame(rows_hier).set_index('Agrupación')

                f_d_hier = {'Var Mercado': fmt_p}
                for cia in cias_obj:
                    f_d_hier[f'Var {cia}'] = fmt_p
                    f_d_hier[f'Veces {cia}'] = fmt_v

                def estilo_jerarquia(data):
                    df_st = pd.DataFrame('', index=data.index, columns=data.columns)
                    for idx, row in data.iterrows():
                        if row['Nivel'] == 0:
                            df_st.loc[idx] = f'background-color: #d1f2eb; font-weight: bold; border-top: 2px solid {COLOR_VERDE}; border-bottom: 2px solid {COLOR_VERDE};'
                        elif row['Nivel'] == 1:
                            df_st.loc[idx] = 'font-weight: bold; background-color: #fcfcfc;'
                        elif row['Nivel'] == 2:
                            df_st.loc[idx] = 'font-style: italic; color: #555;'

                        for col in data.columns:
                            if 'Var' in col and col != 'Nivel':
                                v = pd.to_numeric(row[col], errors='coerce')
                                if pd.notna(v) and v > 0: df_st.loc[idx, col] += ' color: #1E8449;'
                                elif pd.notna(v) and v < 0: df_st.loc[idx, col] += ' color: #E74C3C;'
                            if 'Veces' in col:
                                v_cia = pd.to_numeric(row[col.replace('Veces', 'Var')], errors='coerce')
                                v_mkt = pd.to_numeric(row['Var Mercado'], errors='coerce')
                                if pd.notna(v_cia) and pd.notna(v_mkt):
                                    if v_cia - v_mkt > 0: df_st.loc[idx, col] += ' color: #1E8449; font-weight: bold;'
                                    elif v_cia - v_mkt < 0: df_st.loc[idx, col] += ' color: #E74C3C; font-weight: bold;'
                    return df_st

                st_hier = df_hier.style.format(f_d_hier, na_rep="-").apply(estilo_jerarquia, axis=None).hide(subset=['Nivel'], axis=1).set_table_styles([
                    {'selector': 'th.col_heading', 'props': [('background-color', COLOR_VERDE), ('color', 'white'), ('font-size', '11px')]},
                    {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '11px')]},
                    {'selector': 'th.row_heading', 'props': [('text-align', 'left')]}
                ])
                display(st_hier)

            # --- TOP RIGHT: Base Detalle Regional ---
            with out_ml:
                display(HTML(header_html("BASE DETALLE REGIONAL")))
                if not tab_reg.empty:
                    ord_r = ['TOTAL'] + sorted([r for r in reg_obj if r in tab_reg.index])
                    t_r = tab_reg.loc[ord_r]
                    fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in t_r.columns}
                    display(t_r.style.format(fc, na_rep="-").apply(aplicar_estilos, ord_c=cols_r, lbl_a=lbl_anterior, lbl_h=lbl_actual, axis=None).set_table_styles(tbl_styles))

            # --- MID LEFT: Veces por Ramo ---
            with out_mr:
                display(HTML(header_html("VECES POR RAMO")))
                if not tab_ramo.empty and cias_obj:
                    cia_ref = cias_obj[0]
                    if (cia_ref, 'Veces') in tab_ramo.columns:
                        df_graf = tab_ramo.loc[[r for r in ramos_obj if r in tab_ramo.index]].dropna(subset=[(cia_ref, 'Veces')])
                        if not df_graf.empty:
                            fig, ax = plt.subplots(figsize=(6, 2.5))
                            y_pos = np.arange(len(df_graf))
                            vals = df_graf[(cia_ref, 'Veces')].values
                            colors = ['#27AE60' if v >= 1 else '#145A32' for v in vals]
                            ax.barh(y_pos, vals, align='center', color=colors, height=0.5)
                            ax.set_yticks(y_pos)
                            ax.set_yticklabels([str(x)[:12] for x in df_graf.index], fontsize=8)
                            ax.invert_yaxis()
                            ax.set_xlabel('Veces', fontsize=8)
                            for i, v in enumerate(vals):
                                ax.text(v + 0.1, i, f'{v:.2f}x', va='center', fontsize=8, color='gray')
                            ax.spines['top'].set_visible(False)
                            ax.spines['right'].set_visible(False)
                            plt.tight_layout()
                            plt.show()

            # --- MID RIGHT: % Var por Año y Ramo ---
            with out_bl:
                display(HTML(header_html("% VAR POR AÑO Y RAMO")))
                if not tab_ramo.empty:
                    ord_rm = ['TOTAL'] + sorted([r for r in ramos_obj if r in tab_ramo.index])
                    t_rm = tab_ramo.loc[ord_rm]
                    fc = {c: fmt_m if c[1] in [lbl_anterior, lbl_actual] else (fmt_p if c[1]=='Var' else fmt_v) for c in t_rm.columns}
                    display(t_rm.style.format(fc, na_rep="-").apply(aplicar_estilos, ord_c=cols_rm, lbl_a=lbl_anterior, lbl_h=lbl_actual, axis=None).set_table_styles(tbl_styles))

            # --- BOTTOM: Ranking Global de Compañías ---
            with out_br:
                display(HTML(header_html("RANKING GLOBAL DE CRECIMIENTO POR COMPAÑÍA")))
                # Filtrar base solo para calcular el mercado de las regiones y ramos seleccionados
                df_rk_base = df_base.groupby(['COMPANIA', 'ETIQUETA_TIEMPO'])['TOTAL'].sum().unstack(fill_value=0)

                if lbl_actual in df_rk_base.columns and lbl_anterior in df_rk_base.columns:
                    df_rk_base['Crecimiento (%)'] = (df_rk_base[lbl_actual] / df_rk_base[lbl_anterior]) - 1
                    df_rk_base.replace([np.inf, -np.inf], np.nan, inplace=True)

                    df_rk_base = df_rk_base.sort_values(lbl_actual, ascending=False)
                    df_rk_fin = df_rk_base[[lbl_actual, 'Crecimiento (%)']].rename(columns={lbl_actual: 'Primas Actuales'})

                    # Agregar el Mercado Global (Suma de todas las compañías para el filtro actual)
                    mkt_ant_rk = df_rk_base[lbl_anterior].sum()
                    mkt_act_rk = df_rk_fin['Primas Actuales'].sum()
                    mkt_crec = (mkt_act_rk / mkt_ant_rk - 1) if mkt_ant_rk != 0 else np.nan

                    df_rk_fin.loc['*** TOTAL MERCADO ***'] = [mkt_act_rk, mkt_crec]

                    def c_rk(s): return ['color: #1E8449; font-weight: bold;' if v > 0 else 'color: #E74C3C; font-weight: bold;' if v < 0 else 'color: gray;' for v in s]
                    def b_total(data):
                        df_st = pd.DataFrame('', index=data.index, columns=data.columns)
                        if '*** TOTAL MERCADO ***' in df_st.index:
                            df_st.loc['*** TOTAL MERCADO ***'] = f'background-color: #d1f2eb; font-weight: bold; border-top: 2px solid {COLOR_VERDE}; color: black;'
                        return df_st

                    st_rk = df_rk_fin.style.format({'Primas Actuales': fmt_m, 'Crecimiento (%)': fmt_p}, na_rep="-")\
                            .apply(c_rk, subset=['Crecimiento (%)'])\
                            .apply(b_total, axis=None)\
                            .set_table_styles([
                                {'selector': 'th', 'props': [('background-color', COLOR_VERDE), ('color', 'white'), ('font-size', '11px'), ('text-align', 'center')]},
                                {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '11px')]},
                                {'selector': 'th.row_heading', 'props': [('text-align', 'left'), ('background-color', '#f4f4f4'), ('color', 'black')]}
                            ])
                    display(st_rk)

            # Ensamblar el Grid
            grid = widgets.VBox([
                widgets.HBox([out_hier, out_ml]),
                widgets.HBox([out_mr, out_bl]),
                out_br
            ])
            display(Markdown(f"### 📈 Reporte Analítico: {fecha_fin.year} vs {fecha_fin_ant.year}"))
            display(grid)
            display(Markdown("<br><hr><br>"))

# 5. Renderizado UI
# ------------------------------------------------------------------------------
ui_tiempo = widgets.VBox([widgets.HBox([dropdown_anio_inicio, dropdown_mes_inicio]), widgets.HBox([dropdown_anio_fin, dropdown_mes_fin, widgets.Label(" | "), input_meses, widgets.Label(" | "), select_anios_atras]), mensaje_alerta], layout={'border': '1px solid #ddd', 'padding': '10px', 'margin': '10px 0'})
ui_datos = widgets.HBox([select_ramos, select_companias, select_regionales])
ui = widgets.VBox([ui_tiempo, ui_datos, widgets.VBox([boton_generar], layout={'align_items': 'center', 'margin': '15px 0'})])
boton_generar.on_click(generar_tablero_maestro)
display(Markdown("---"))
display(ui, out_main)

---

Output()